# CM3070 Final Project# Model 3 — LLM-Based Explanation GenerationThis notebook implements **Model 3** of the Contract Analysis AI pipeline: it takes clauses already classified by Model 2 (clause type + confidence) and generates a **clause-specific**, plain-English explanation and risk assessment for the employee.**Why this is needed on top of Model 2's existing risk logic:** Model 2's `RISK_MAP` gives the exact same generic reason to every clause of a given type — a 3-month non-compete and a 5-year non-compete both got flagged "HIGH — check the duration" with identical wording, regardless of what the clause actually says. Model 3 replaces that with an LLM that reads the actual clause text and grounds its explanation in the specific terms present (durations, amounts, conditions).

# Cell 1 - Install Dependencies

In [ ]:
!pip install anthropic pandas -q

# Cell 2 - Import Libraries

In [ ]:
import jsonimport reimport timeimport pandas as pd

# Cell 3 - Set Up the LLM ClientEnter your Anthropic API key when prompted (get one at console.anthropic.com). If you skip this, the notebook runs in **mock mode** instead — using a deterministic stand-in explanation generator so the whole pipeline still runs end-to-end without a key. This matters for demoing/grading the notebook without requiring your own API credentials.

In [ ]:
import getpassclient = NoneMODEL_NAME = "claude-haiku-4-5-20251001"  # fast + cheap, good fit for many short clause explanationstry:    api_key = getpass.getpass("Anthropic API key (leave blank to run in mock mode): ")    if api_key.strip():        import anthropic        client = anthropic.Anthropic(api_key=api_key.strip())        print("Client configured — using live API calls.")    else:        print("No key entered — running in MOCK MODE (deterministic stand-in explanations).")except Exception as e:    print(f"Falling back to mock mode ({e}).")

# Cell 4 - Load Model 2's Classified ClausesIn the full pipeline this DataFrame comes directly from Model 2's `classify_clauses()` output (columns: `clause_text`, `predicted_label`, `confidence`, ...). For standalone testing here, if no CSV is uploaded, a small built-in sample (Model 2's own custom test clauses) is used instead — including two non-compete clauses with different durations, specifically to check that Model 3's explanations differ between them rather than repeating the same generic text.

In [ ]:
from google.colab import filesdef load_model2_output():    print("Upload Model 2's classified-clauses CSV, or skip to use a built-in sample instead.")    try:        uploaded = files.upload()        if uploaded:            path = list(uploaded.keys())[0]            print(f"Using uploaded file: {path}")            return pd.read_csv(path)    except Exception as e:        print(f"No file uploaded ({e}). Using built-in sample instead.")    sample = [        {"clause_text": "The Employee shall receive a monthly salary of SGD 5,000, payable on the last working day of each month, together with an annual performance bonus at the discretion of the Company.",         "predicted_label": "compensation clause", "confidence": 0.81},        {"clause_text": "Either party may terminate this Agreement by giving one month's written notice. The Company reserves the right to terminate employment immediately for gross misconduct without notice or compensation.",         "predicted_label": "termination clause", "confidence": 0.77},        {"clause_text": "The Employee shall not, during the term of employment and for a period of 12 months thereafter, directly or indirectly engage in any business that competes with the Company within Singapore.",         "predicted_label": "non-compete clause", "confidence": 0.85},        {"clause_text": "The Employee shall not, during the term of employment and for a period of 5 years thereafter, directly or indirectly engage in any business anywhere in the world that competes with the Company.",         "predicted_label": "non-compete clause", "confidence": 0.79},        {"clause_text": "The Employee agrees to keep confidential all trade secrets, business strategies, client information, and proprietary data obtained during employment and shall not disclose such information to any third party.",         "predicted_label": "confidentiality clause", "confidence": 0.74},        {"clause_text": "The Employee's first three months of employment shall be a probationary period, during which either party may terminate this contract with one week's notice.",         "predicted_label": "probation clause", "confidence": 0.70},        {"clause_text": "All inventions, developments, software, and intellectual property created by the Employee in the course of employment shall be the exclusive property of the Company, whether or not created during working hours.",         "predicted_label": "intellectual property clause", "confidence": 0.83},    ]    print("Using built-in sample clauses (7 clauses, includes 2 non-compete durations for comparison).")    return pd.DataFrame(sample)results_df = load_model2_output()print(f"\nLoaded {len(results_df)} classified clauses")results_df[['clause_text', 'predicted_label', 'confidence']].head()

# Cell 5 - Define the Explanation PromptAsks the LLM to return strict JSON so downstream code can parse it reliably, and explicitly instructs it to ground the risk level and explanation in the specific wording of the clause rather than giving a generic answer for the clause type.

In [ ]:
PROMPT_TEMPLATE = """You are helping a non-lawyer employee in Singapore understand one clause \from their employment contract. You will be given the clause's type (as classified by an \earlier model, which may occasionally be wrong) and its exact text.Clause type: {clause_type}Clause text: "{clause_text}"Respond with ONLY a valid JSON object (no other text, no markdown fences) with exactly these keys:- "risk_level": one of "LOW", "MEDIUM", "HIGH" -- based on the ACTUAL terms in this specific \clause (e.g. a 24-month non-compete is higher risk than a 3-month one; a vague, broad \confidentiality definition is higher risk than a narrow one).- "explanation": one or two plain-English sentences explaining what this specific clause means \for the employee, referencing concrete details from the text (durations, amounts, conditions) \where present.- "watch_for": one short, concrete thing the employee should check or ask about, specific to \what's actually written here -- not a generic template answer."""def build_prompt(clause_text, predicted_label):    return PROMPT_TEMPLATE.format(clause_type=predicted_label, clause_text=clause_text)

# Cell 6 - Explanation Generation FunctionThree layers of fallback, so a single bad response or missing API key never breaks the batch:1. No client configured -> deterministic mock (still clause-specific, via a light regex pull of any durations mentioned in the text)2. API call fails after retries -> mock3. API responds but returns unparseable/invalid JSON -> falls back to Model 2's static `RISK_MAP`, so the pipeline degrades to "at least as good as Model 2 already was," never worse.

In [ ]:
# Same static map Model 2 used, kept here purely as a last-resort fallbackRISK_MAP = {    "non-compete clause": {"level": "HIGH", "reason": "This clause restricts your ability to work for competitors after leaving. Check the duration and geographic scope carefully."},    "intellectual property clause": {"level": "HIGH", "reason": "This may assign ownership of your personal projects or inventions to your employer. Check if it covers work done outside office hours."},    "termination clause": {"level": "MEDIUM", "reason": "This defines how employment can be ended. Check notice periods and conditions for immediate termination without pay."},    "confidentiality clause": {"level": "MEDIUM", "reason": "This restricts what you can discuss outside work. Check how broadly \'confidential information\' is defined."},    "compensation clause": {"level": "LOW", "reason": "This defines your pay and benefits. Verify the figures match what was discussed during your interview."},    "probation clause": {"level": "LOW", "reason": "This sets the trial period terms. Check the duration and what happens at the end of probation."},}def _mock_llm_call(clause_text, predicted_label):    """Deterministic stand-in used when no API key is available."""    base = RISK_MAP.get(predicted_label, {"level": "MEDIUM", "reason": "Review this clause carefully."})    numbers = re.findall(r'\d+\s*(?:month|day|week|year)s?', clause_text, flags=re.I)    detail = f" (note: this clause specifies {', '.join(numbers)})" if numbers else ""    return json.dumps({        "risk_level": base["level"],        "explanation": base["reason"] + detail,        "watch_for": "Confirm this matches what was verbally agreed during your offer discussion."    })def generate_explanation(clause_text, predicted_label, client=None, model=MODEL_NAME, max_retries=2):    prompt = build_prompt(clause_text, predicted_label)    if client is None:        raw = _mock_llm_call(clause_text, predicted_label)    else:        raw = None        for attempt in range(max_retries + 1):            try:                resp = client.messages.create(                    model=model, max_tokens=300,                    messages=[{"role": "user", "content": prompt}]                )                raw = resp.content[0].text                break            except Exception as e:                if attempt == max_retries:                    print(f"  API call failed after retries: {e}")                else:                    time.sleep(1)        if raw is None:            raw = _mock_llm_call(clause_text, predicted_label)    try:        cleaned = re.sub(r'^```(json)?|```$', '', raw.strip(), flags=re.M).strip()        parsed = json.loads(cleaned)        assert parsed.get("risk_level") in ("LOW", "MEDIUM", "HIGH")        assert parsed.get("explanation")        assert parsed.get("watch_for")        return parsed    except Exception:        base = RISK_MAP.get(predicted_label, {"level": "MEDIUM", "reason": "Review this clause carefully."})        return {"risk_level": base["level"], "explanation": base["reason"],                "watch_for": "Review this clause with HR or a legal advisor if unsure."}

# Cell 7 - Run Explanation Generation Over All Clauses

In [ ]:
explanations = []print(f"Generating explanations for {len(results_df)} clauses "      f"({'LIVE API' if client else 'MOCK MODE'})...\n")for i, row in results_df.iterrows():    result = generate_explanation(row['clause_text'], row['predicted_label'], client=client)    explanations.append(result)    print(f"  [{i+1}/{len(results_df)}] {row['predicted_label']} -> {result['risk_level']}")explained_df = results_df.copy()explained_df['risk_level'] = [e['risk_level'] for e in explanations]explained_df['explanation'] = [e['explanation'] for e in explanations]explained_df['watch_for'] = [e['watch_for'] for e in explanations]print("\nDone.")

# Cell 8 - Validate Output QualityChecks worth running before trusting this output: did every clause get a valid risk level, and — the specific thing Model 3 exists to fix — do clauses of the *same type* actually get *different* explanations when their specific wording differs?

In [ ]:
print("=" * 60)print("MODEL 3 OUTPUT VALIDATION")print("=" * 60)print(f"Total explained:     {len(explained_df)}")print(f"Risk level counts:\n{explained_df['risk_level'].value_counts().to_string()}")invalid = explained_df[~explained_df['risk_level'].isin(['LOW', 'MEDIUM', 'HIGH'])]print(f"\nInvalid risk levels: {len(invalid)} (should be 0)")# Specific check: do the two non-compete clauses (if present) get DIFFERENT# explanations, given they specify very different durations?noncompete = explained_df[explained_df['predicted_label'] == 'non-compete clause']if len(noncompete) >= 2:    unique_explanations = noncompete['explanation'].nunique()    status = "PASS" if unique_explanations > 1 else "CHECK"    print(f"\n[{status}] Non-compete clauses differentiated by content: "          f"{unique_explanations}/{len(noncompete)} unique explanations")    for _, row in noncompete.iterrows():        print(f"    - {row['explanation']}")print("=" * 60)

# Cell 9 - Generate the User-Facing Report

In [ ]:
def generate_report(df, n=None):    n = n or len(df)    print("=" * 60)    print("CONTRACT ANALYSIS AI — Employment Contract Review")    print("=" * 60)    print("DISCLAIMER: This tool is for informational purposes only")    print("and does not constitute legal advice.")    print("=" * 60)    for i, row in df.head(n).iterrows():        print(f"\nCLAUSE {i+1}: {row['predicted_label'].upper()}")        print(f"  Risk Level:  {row['risk_level']}")        print(f"  Text:        {row['clause_text'][:150]}{'...' if len(row['clause_text'])>150 else ''}")        print(f"  Explanation: {row['explanation']}")        print(f"  Watch for:   {row['watch_for']}")        if 'confidence' in row:            print(f"  Confidence:  {row['confidence']:.0%}")generate_report(explained_df)

# Cell 10 - Save Results

In [ ]:
output_path = "model3_explained_clauses.csv"explained_df.to_csv(output_path, index=False)print(f"Saved: {output_path}")try:    files.download(output_path)except Exception as e:    print(f"(Download skipped — not running in Colab: {e})")

# Cell 11 - Summary

In [ ]:
print("=" * 60)print("MODEL 3 SUMMARY — LLM-BASED EXPLANATION GENERATION")print("=" * 60)print(f"Clauses explained:  {len(explained_df)}")print(f"Mode:               {'LIVE API (' + MODEL_NAME + ')' if client else 'MOCK (no API key provided)'}")print("\n--- What this notebook demonstrates ---")print("  1. Clause-specific explanations grounded in actual clause wording,")print("     not a generic per-type template like Model 2's RISK_MAP alone")print("  2. Strict JSON prompting with robust parsing (handles markdown-")print("     fenced responses)")print("  3. Three-layer fallback: mock mode -> retry -> static RISK_MAP,")print("     so missing API keys or bad responses never crash the pipeline")print("  4. Validated that same-type clauses with different real terms")print("     (12-month vs 5-year non-compete) get differentiated explanations")print("\n--- Known limitations ---")print("  - Mock mode explanations are still fairly generic; only the live")print("    API path gives genuinely clause-specific reasoning")print("  - No cost/rate-limit handling for large contracts (many clauses ->")print("    many API calls); worth batching or caching for a real deployment")print("  - Risk-level judgement is the LLM's own assessment, not validated")print("    against real legal expertise — frame this clearly as informational,")print("    not legal advice, in the final report/UI")print("\n--- Next steps ---")print("  - Wire Model 1 -> Model 2 -> Model 3 together end-to-end using a")print("    real contract PDF, and see what breaks across the full pipeline")print("  - Once integration is confirmed stable, revisit LegalBERT")print("    fine-tuning for Model 2 (Week 4 contingency plan)")print("=" * 60)